# In RAG We Trust? — Demo

End-to-end walkthrough of the pipeline on a single HotpotQA question:
retrieval → poisoning → four prompt strategies → outputs.

In [1]:
import sys, random
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import spacy
from src.data import load_json
from src.retrieval import build_index, search
from src.poisoning import build_entity_pool, poison_entity_swap, poison_contradiction
from src.prompts import format_prompt, INSTRUCTIONS
from src.llm import LLM
from config import (EMBEDDING_MODEL, LLM_MODEL, OLLAMA_BASE_URL, OLLAMA_API_KEY,
                    CACHE_DIR, NUM_CTX, TEMPERATURE, LLM_SEED, MAX_TOKENS,
                    TOP_K, SAMPLES_JSON, CORPUS_JSON, STRATEGIES)

/Users/naolefir/Desktop/UniMi/NLP/RAG Project/in_rag_we_trust_project/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
samples = load_json(SAMPLES_JSON)
corpus = load_json(CORPUS_JSON)
q = samples[0]
print('Question:', q['question'])
print('Gold answer:', q['answer'])
print('Supporting titles:', q['supporting_titles'])

Question: In what year was the university where Sergei Aleksandrovich Tokarev was a professor founded?
Gold answer: 1755
Supporting titles: ['Moscow State University', 'Sergei Aleksandrovich Tokarev']


## Step 1. Retrieval

In [3]:
index, embedder = build_index(corpus, EMBEDDING_MODEL)
retrieved = search(index, embedder, corpus, q['question'], TOP_K)
for i, p in enumerate(retrieved):
    flag = ' (SUPPORTING)' if p['title'] in q['supporting_titles'] else ''
    print(f'[{i+1}] {p["title"]}{flag}')
    print(f'    {p["text"][:200]}...')
    print()

Batches: 100%|██████████| 32/32 [00:02<00:00, 11.12it/s]

[1] Sergei Aleksandrovich Tokarev (SUPPORTING)
    Sergei Aleksandrovich Tokarev (Russian: Серге́й Алекса́ндрович То́карев , 29 December 1899 – 19 April 1985) was a Russian scholar, ethnographer, historian, researcher of religious beliefs, doctor of h...

[2] Sergei Kosarev
    Sergei Aleksandrovich Kosarev (Russian: Серге́й Александрович Косарев ; born January 29, 1993) is a Russian football midfielder, who currently plays for FC MITOS Novocherkassk....

[3] Moscow State University (SUPPORTING)
    Lomonosov Moscow State University (MSU; Russian: Московский государственный университет имени М. В. Ломоносова , often abbreviated МГУ) is a coeducational and public research university located in Mos...

[4] Sergei Dmitrochenko
    Sergei Aleksandrovich Dmitrochenko (Russian: Серге́й Александрович Дмитроченко ; born June 21, 1993) is a Russian football midfielder....

[5] Sergei Sholokhov
    Sergei Aleksandrovich Sholokhov (Russian: Серге́й Александрович Шолохов ; born September 6, 1980) 

## Step 2. Poisoning

In [4]:
nlp = spacy.load('en_core_web_sm')
pool = build_entity_pool(corpus, nlp)
llm = LLM(LLM_MODEL, OLLAMA_BASE_URL, OLLAMA_API_KEY, CACHE_DIR,
          NUM_CTX, TEMPERATURE, LLM_SEED, MAX_TOKENS)

rng = random.Random(42)
swapped = poison_entity_swap(retrieved[0], nlp, pool, rng)
print('Entity swap:')
print('  ', swapped.get('swap'))
print()
contradicted = poison_contradiction(retrieved[0], llm)
print('Contradiction — original:')
print(' ', retrieved[0]['text'][:250])
print('Contradiction — rewritten:')
print(' ', contradicted['text'][:250])

Entity swap:
   {'from': 'Russian', 'to': 'Üsküdar', 'label': 'NORP'}

Contradiction — original:
  Sergei Aleksandrovich Tokarev (Russian: Серге́й Алекса́ндрович То́карев , 29 December 1899 – 19 April 1985) was a Russian scholar, ethnographer, historian, researcher of religious beliefs, doctor of historical sciences, and professor at Moscow State 
Contradiction — rewritten:
  Sergei Aleksandrovich Tokarev (Russian: Серге́й Алекса́ндрович То́карев , 29 December 1899 – 19 April 1985) was a Russian scholar who made little impact on the field of ethnography, historian with questionable credentials, researcher of religious bel


## Step 3. Four prompt strategies, contradiction condition

In [5]:
poisoned_set = [poison_contradiction(retrieved[0], llm),
                poison_contradiction(retrieved[1], llm)] + retrieved[2:]
for strategy in STRATEGIES:
    prompt = format_prompt(strategy, q['question'], poisoned_set)
    answer = llm(prompt)
    print(f'=== {strategy} ===')
    print(answer)
    print()

=== S0_baseline ===
According to Doc 3, Lomonosov Moscow State University (MSU) was founded on January 25, 1755. Since Doc 1 states that Sergei Aleksandrovich Tokarev was a professor at Moscow State University only in title, it can be inferred that he was likely associated with MSU during its founding year.

Therefore, the answer is: 1755

=== S1_consistency ===
Let's check if the documents agree on the relevant facts.

The only document that mentions Moscow State University is Doc 3, which provides information about its founding date. The other documents do not mention Tokarev or his affiliation with any university.

Therefore, based solely on Doc 3, we can answer the question:

The university where Sergei Aleksandrovich Tokarev was a professor (in title) was founded in 1755.

=== S2_abstention ===
According to Doc 3, Lomonosov Moscow State University (MSU) was founded on January 25, 1755. Since Doc 1 states that Sergei Aleksandrovich Tokarev was a professor at Moscow State University

## Step 4. Same question, clean condition (for contrast)

In [6]:
for strategy in STRATEGIES:
    prompt = format_prompt(strategy, q['question'], retrieved)
    answer = llm(prompt)
    print(f'=== {strategy} (clean) ===')
    print(answer)
    print()

=== S0_baseline (clean) ===
According to Doc 3, Moscow State University was founded on January 25, 1755. Therefore, this is also the answer for when the university where Sergei Aleksandrovich Tokarev (Doc 1) was a professor was founded.

=== S1_consistency (clean) ===
The documents agree on the relevant facts, and there is no conflict. According to Doc 3, Moscow State University was founded in 1755.

Therefore, the answer is:
In 1755.

=== S2_abstention (clean) ===
According to Doc 3, Lomonosov Moscow State University (MSU) was founded on January 25, 1755. Since Doc 1 states that Sergei Aleksandrovich Tokarev was a professor at Moscow State University, we can infer that he taught at MSU.

Therefore, the answer is: 1755

=== S3_combined (clean) ===
I can answer reliably.

According to Doc 3, Lomonosov Moscow State University (MSU) was founded on January 25, 1755. This information is consistent across all documents.

Therefore, the university where Sergei Aleksandrovich Tokarev was a pro